# 使用 Milvus 和 DeepSeek 构建 RAG

DeepSeek 帮助开发者使用高性能语言模型构建和扩展 AI 应用。它提供高效的推理、灵活的 API 以及先进的专家混合 (MoE) 架构，用于强大的推理和检索任务。

在本教程中，我们将展示如何使用 Milvus 和 DeepSeek 构建一个检索增强生成 (RAG) 管道。

## 准备工作

### 依赖与环境

In [1]:
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

  Obtaining dependency information for pymilvus[model]==2.5.10 from https://files.pythonhosted.org/packages/b0/4b/847704930ad8ddd0d0975e9a3a5e3fe704f642debe97454135c2b9ee7081/pymilvus-2.5.10-py3-none-any.whl.metadata
  Obtaining dependency information for requests==2.32.3 from https://files.pythonhosted.org/packages/f9/9b/335f9764261e915ed497fcdeb11df5dfd6f7bf257d4a6a2a686d80da4d54/requests-2.32.3-py3-none-any.whl.metadata
  Obtaining dependency information for torch==2.7.0 from https://files.pythonhosted.org/packages/aa/3f/85b56f7e2abcfa558c5fbf7b11eb02d78a4a63e6aeee2bbae3bb552abea5/torch-2.7.0-cp311-none-macosx_11_0_arm64.whl.metadata
  Obtaining dependency information for setuptools>69 from https://files.pythonhosted.org/packages/a3/dc/17031897dae0efacfea57dfd3a82fdd2a2aeb58e0ff71b77b87e44edc772/setuptools-80.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for grpcio<=1.67.1,>=1.49.1 from https://files.pythonhosted.org/packages/b3/9a/e1956f7ca582a22dd1f17b9e26fcb822

---

In [5]:
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("ALIYUN_CLASS_API_KEY")

### 准备数据

我们使用 Milvus 文档 2.4.x 中的 FAQ 页面作为我们 RAG 中的私有知识库，这是一个简单 RAG 管道的良好数据源。

下载 zip 文件并将文档解压到 `milvus_docs` 文件夹。

**建议在命令行执行下面命令**

In [6]:
#!wget https://github.com/milvus-io/milvus-docs/releases/download/v2.4.6-preview/milvus_docs_2.4.x_en.zip
#!unzip -q milvus_docs_2.4.x_en.zip -d milvus_docs

我们从 `milvus_docs/en/faq` 文件夹加载所有 markdown 文件。对于每个文档，我们简单地使用 "# " 来分割文件中的内容，这样可以大致分离出 markdown 文件中每个主要部分的内容。

In [373]:
from glob import glob

text_lines = []

for file_path in glob("milvus_docs/en/faq/*.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

In [374]:
len(text_lines)

72

### 准备 LLM 和 Embedding 模型

DeepSeek 支持 OpenAI 风格的 API，您可以使用相同的 API 进行微小调整来调用 LLM。

In [375]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",  # Aliyun百炼 API 的基地址
)

定义一个 embedding 模型，使用 `milvus_model` 来生成文本嵌入。我们以 `DefaultEmbeddingFunction` 模型为例，这是一个预训练的轻量级嵌入模型。

In [376]:
# from pymilvus import model as milvus_model

# embedding_model = milvus_model.DefaultEmbeddingFunction()

from pymilvus import model as milvus_model

# OpenAI国内代理 https://api.apiyi.com/token 
embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
    model_name='text-embedding-3-large', # Specify the model name
    api_key='sk-XXX', # Provide your OpenAI API key
    base_url='https://api.apiyi.com/v1',
    dimensions=512
)

生成一个测试嵌入并打印其维度和前几个元素。

In [377]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

768
[-0.04836056  0.07163018 -0.01130064 -0.03789344 -0.03320646 -0.01318444
 -0.03041711 -0.02269505 -0.02317867 -0.00426023]


In [378]:
test_embedding_0 = embedding_model.encode_queries(["That is a test"])[0]
print(test_embedding_0[:10])

[-0.0275297   0.06088526  0.00388529 -0.00215193 -0.02774976 -0.01186187
 -0.04020914 -0.06023425 -0.03813157  0.01002724]


## 将数据加载到 Milvus

### 创建 Collection

In [379]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./milvus_demo.db")

collection_name = "my_rag_collection"

/root/anaconda3/envs/deepseek/lib/python3.13/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


关于 `MilvusClient` 的参数：

*   将 `uri` 设置为本地文件，例如 `./milvus.db`，是最方便的方法，因为它会自动利用 Milvus Lite 将所有数据存储在此文件中。
*   如果您有大规模数据，可以在 Docker 或 Kubernetes 上设置性能更高的 Milvus 服务器。在此设置中，请使用服务器 URI，例如 `http://localhost:19530`，作为您的 `uri`。
*   如果您想使用 Zilliz Cloud（Milvus 的完全托管云服务），请调整 `uri` 和 `token`，它们对应 Zilliz Cloud 中的 Public Endpoint 和 Api key。

检查 collection 是否已存在，如果存在则删除它。

In [380]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

创建一个具有指定参数的新 collection。

如果我们不指定任何字段信息，Milvus 将自动创建一个默认的 `id` 字段作为主键，以及一个 `vector` 字段来存储向量数据。一个保留的 JSON 字段用于存储非 schema 定义的字段及其值。

`metric_type` (距离度量类型):
     作用：定义如何计算向量之间的相似程度。
     例如：`IP` (内积) - 值越大通常越相似；`L2` (欧氏距离) - 值越小越相似；`COSINE` (余弦相似度) - 通常转换为距离，值越小越相似。
     选择依据：根据你的嵌入模型的特性和期望的相似性定义来选择。

 `consistency_level` (一致性级别):
     作用：定义数据写入后，读取操作能多快看到这些新数据。
     例如：
         `Strong` (强一致性): 总是读到最新数据，可能稍慢。
         `Bounded` (有界过期): 可能读到几秒内旧数据，性能较好 (默认)。
         `Session` (会话一致性): 自己写入的自己能立刻读到。
         `Eventually` (最终一致性): 最终会读到新数据，但没时间保证，性能最好。
     选择依据：在数据实时性要求和系统性能之间做权衡。

简单来说：
 `metric_type`：怎么算相似。
 `consistency_level`：新数据多久能被读到。

In [381]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

### 插入数据

遍历文本行，创建嵌入，然后将数据插入 Milvus。

这里有一个新字段 `text`，它是在 collection schema 中未定义的字段。它将自动添加到保留的 JSON 动态字段中，该字段在高级别上可以被视为普通字段。

In [382]:
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 72/72 [00:00<00:00, 1597830.10it/s]


{'insert_count': 72, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71], 'cost': 0}

## 构建 RAG

### 检索查询数据

我们指定一个关于 Milvus 的常见问题。

In [383]:
question = "How is data stored in milvus?"

在 collection 中搜索该问题，并检索语义上最匹配的前3个结果。

In [384]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)

让我们看一下查询的搜索结果

In [385]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        " Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###",
        0.6572662591934204
    ],
    [
        "How does Milvus flush data?\n\nMilvus returns success when inserted data are loaded to t

### 使用 LLM 获取 RAG 响应

将检索到的文档转换为字符串格式。

In [386]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [387]:
context

" Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data types and precision?\n\nMilvus supports Binary, Float32, Float16, and BFloat16 vector types.\n\n- Binary vectors: Store

In [388]:
question

'How is data stored in milvus?'

为语言模型定义系统和用户提示。此提示是使用从 Milvus 检索到的文档组装而成的。

In [389]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。
<context>
{context}
</context>
<question>
{question}
</question>
<translated>
</translated>
"""

In [390]:
USER_PROMPT

"\n请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。最后追加原始回答的中文翻译，并用 <translated>和</translated> 标签标注。\n<context>\n Where does Milvus store data?\n\nMilvus deals with two types of data, inserted data and metadata. \n\nInserted data, including vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental log. Milvus supports multiple object storage backends, including [MinIO](https://min.io/), [AWS S3](https://aws.amazon.com/s3/?nc1=h_ls), [Google Cloud Storage](https://cloud.google.com/storage?hl=en#object-storage-for-companies-of-all-sizes) (GCS), [Azure Blob Storage](https://azure.microsoft.com/en-us/products/storage/blobs), [Alibaba Cloud OSS](https://www.alibabacloud.com/product/object-storage-service), and [Tencent Cloud Object Storage](https://www.tencentcloud.com/products/cos) (COS).\n\nMetadata are generated within Milvus. Each Milvus module has its own metadata that are stored in etcd.\n\n###\nHow does Milvus handle vector data typ

使用 DeepSeek 提供的 `deepseek-chat` 模型根据提示生成响应。

In [391]:
response = deepseek_client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

Milvus stores data by separating it into two types: inserted data and metadata. 

- **Inserted data**, such as vector data, scalar data, and collection-specific schema, are stored in persistent storage as incremental logs. Milvus supports multiple object storage backends, including MinIO, AWS S3, Google Cloud Storage (GCS), Azure Blob Storage, Alibaba Cloud OSS, and Tencent Cloud COS.

- **Metadata**, which are generated internally by Milvus, are stored in etcd. Each Milvus module maintains its own metadata in etcd for coordination and management purposes.

<translated>
Milvus 通过将数据分为插入数据和元数据来存储数据。

- **插入数据**（如向量数据、标量数据和集合特定的模式）作为增量日志存储在持久化存储中。Milvus 支持多种对象存储后端，包括 MinIO、AWS S3、Google Cloud Storage (GCS)、Azure Blob Storage、阿里云 OSS 和腾讯云 COS。

- **元数据** 由 Milvus 内部生成，并存储在 etcd 中。每个 Milvus 模块都会在 etcd 中维护自己的元数据，用于协调和管理。
</translated>


## 民法典

In [29]:
from glob import glob

text_lines = []

data_text = ""
#chapter_stack = [''] ##定义一个空字符串，作为h1
for file_path in glob("mfd.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()
    simple_split = file_text.split('\n')
#    for line in simple_split:
#        ## 按MD结构进行处理，生成章节栈，如果章节变更则更换text_lines
#        if '#' in line:
#            c_depth = line.count('#')
#            stack_depth = len(chapter_stack)
#            ## 章节结构展开的情况直接入栈
#            if c_depth > stack_depth:
#                chapter_stack.append(line)
#            else:
#                ##变更章节栈，初始化data_text
#                if data_text:
#                    text_lines.append(data_text)
#                while len(chapter_stack) >= c_depth:
#                    chapter_stack.pop()
#                chapter_stack.append(line)
#                data_text = '\n'.join(chapter_stack)
#        elif line:
#            if not data_text:
#                data_text = '\n'.join(chapter_stack)
#            if '*' in line:
#                data_text = data_text + '\n'
#            data_text = data_text + line
#    if data_text:
#        text_lines.append(data_text)
    for line in simple_split:
        ##按空行分段
        if not line:
            if data_text:
                text_lines.append(data_text)
            data_text = ""
        elif "-" in line:
            continue
        else:
            data_text = data_text + line
    if data_text:
        text_lines.append(data_text)

In [30]:
text_lines[:10]

['## 中华人民共和国民法典',
 '### （二）物权编',
 '#### 第一章 一般规定',
 '**第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。',
 '**第二百零五条** 本编调整因物的归属和利用产生的民事关系。',
 '**第二百零六条** 国家坚持和完善社会主义公有制为主体、多种所有制经济共同发展的基本经济制度。国家巩固和发展公有制经济，鼓励、支持和引导非公有制经济的发展。国家实行社会主义市场经济，保障一切市场主体的平等法律地位和发展权利。',
 '**第二百零七条** 国家、集体、私人的物权和其他权利人的物权受法律平等保护，任何组织或者个人不得侵犯。',
 '**第二百零八条** 不动产权利的设立、变更、转让和消灭，应当依照法律规定登记。动产物权的设立和转让，应当依照法律规定交付。',
 '**第二百零九条** 不动产物权的设立、变更、转让和消灭，经依法登记，发生效力；未经登记，不发生效力，但是法律另有规定的除外。依法属于国家所有的自然资源，所有权可以不登记。',
 '**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。']

In [31]:
len(text_lines)

416

In [32]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",  # Aliyun百炼 API 的基地址
)

In [33]:
!pip install sentence_transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [34]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('BAAI/bge-large-zh-v1.5')

In [35]:
test_embedding = embedding_model.encode(["让我们说中文"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

1024
[ 0.00621221  0.00178695 -0.02969557 -0.0015171   0.03116539  0.01011374
 -0.06609055  0.00949357 -0.02912799  0.01765241]


In [36]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./mfd.db")

collection_name = "mfd_collection"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [37]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [38]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP", 
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

In [39]:
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 416/416 [00:00<00:00, 3851722.88it/s]


{'insert_count': 416, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 

## Question1

In [40]:
question1 = "不动产登记,由哪里的机构办理" 

In [41]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode(
        [question1]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)

In [42]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "**\u7b2c\u4e8c\u767e\u4e00\u5341\u6761** \u4e0d\u52a8\u4ea7\u767b\u8bb0\uff0c\u7531\u4e0d\u52a8\u4ea7\u6240\u5728\u5730\u7684\u767b\u8bb0\u673a\u6784\u529e\u7406\u3002\u56fd\u5bb6\u5bf9\u4e0d\u52a8\u4ea7\u5b9e\u884c\u7edf\u4e00\u767b\u8bb0\u5236\u5ea6\u3002\u7edf\u4e00\u767b\u8bb0\u7684\u8303\u56f4\u3001\u767b\u8bb0\u673a\u6784\u548c\u767b\u8bb0\u529e\u6cd5\uff0c\u7531\u6cd5\u5f8b\u3001\u884c\u653f\u6cd5\u89c4\u89c4\u5b9a\u3002",
        0.7171206474304199
    ],
    [
        "**\u7b2c\u4e8c\u767e\u4e00\u5341\u4e8c\u6761** \u767b\u8bb0\u673a\u6784\u5e94\u5f53\u5c65\u884c\u4e0b\u5217\u804c\u8d23\uff1a\uff08\u4e00\uff09\u5ba1\u67e5\u7533\u8bf7\u4eba\u63d0\u4f9b\u7684\u6750\u6599\uff1b\uff08\u4e8c\uff09\u8be2\u95ee\u7533\u8bf7\u4eba\uff1b\uff08\u4e09\uff09\u5982\u5b9e\u3001\u53ca\u65f6\u767b\u8bb0\uff1b\uff08\u56db\uff09\u6cd5\u5f8b\u3001\u884c\u653f\u6cd5\u89c4\u89c4\u5b9a\u7684\u5176\u4ed6\u804c\u8d23\u3002\u7533\u8bf7\u767b\u8bb0\u7684\u4e0d\u52a8\u4ea7\u5b58\u5728\u5

In [43]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [44]:
context

'**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。\n**第二百一十二条** 登记机构应当履行下列职责：（一）审查申请人提供的材料；（二）询问申请人；（三）如实、及时登记；（四）法律、行政法规规定的其他职责。申请登记的不动产存在尚未解决的权属争议的，登记机构应当不予登记，并书面告知申请人。\n**第二百一十五条** 不动产登记簿由登记机构管理。不动产登记簿应当采用纸质形式或者电子形式。不动产登记簿采用电子形式的，应当备份。'

In [45]:
question1

'不动产登记,由哪里的机构办理'

In [46]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文法律条文中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的法律条文来回答用 <question> 标签括起来的问题。最后追加你引用的法律条文，具体到哪一条以及那一条的原文是什么。
<context>
{context}
</context>
<question>
{question1}
</question>
"""

In [47]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的法律条文来回答用 <question> 标签括起来的问题。最后追加你引用的法律条文，具体到哪一条以及那一条的原文是什么。\n<context>\n**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。\n**第二百一十二条** 登记机构应当履行下列职责：（一）审查申请人提供的材料；（二）询问申请人；（三）如实、及时登记；（四）法律、行政法规规定的其他职责。申请登记的不动产存在尚未解决的权属争议的，登记机构应当不予登记，并书面告知申请人。\n**第二百一十五条** 不动产登记簿由登记机构管理。不动产登记簿应当采用纸质形式或者电子形式。不动产登记簿采用电子形式的，应当备份。\n</context>\n<question>\n不动产登记,由哪里的机构办理\n</question>\n'

In [48]:
response = deepseek_client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

不动产登记由不动产所在地的登记机构办理。

引用法律条文：  
**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。


## Question2

In [49]:
# question2 = "耕地的承包期是多久"
question2 = "李某与张某签订了一份房屋买卖合同，约定李某将一套房屋出售给张某，但李某在签订合同后反悔，拒绝履行合同。李某的行为是否违法?"

In [50]:
question2

'李某与张某签订了一份房屋买卖合同，约定李某将一套房屋出售给张某，但李某在签订合同后反悔，拒绝履行合同。李某的行为是否违法?'

In [51]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode(
        [question2]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  
    output_fields=["text"],  # 返回 text 字段
)

In [52]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "**\u7b2c\u4e94\u767e\u4e03\u5341\u4e00\u6761** \u5f53\u4e8b\u4eba\u4e00\u65b9\u4e0d\u5c65\u884c\u5408\u540c\u4e49\u52a1\u6216\u8005\u5c65\u884c\u5408\u540c\u4e49\u52a1\u4e0d\u7b26\u5408\u7ea6\u5b9a\u7684\uff0c\u5e94\u5f53\u627f\u62c5\u7ee7\u7eed\u5c65\u884c\u3001\u91c7\u53d6\u8865\u6551\u63aa\u65bd\u6216\u8005\u8d54\u507f\u635f\u5931\u7b49\u8fdd\u7ea6\u8d23\u4efb\u3002",
        0.7275442481040955
    ],
    [
        "**\u7b2c\u4e94\u767e\u516d\u5341\u4e00\u6761** \u5f53\u4e8b\u4eba\u4e00\u65b9\u4e0d\u5c65\u884c\u5408\u540c\u4e49\u52a1\u6216\u8005\u5c65\u884c\u5408\u540c\u4e49\u52a1\u4e0d\u7b26\u5408\u7ea6\u5b9a\u7684\uff0c\u5bf9\u65b9\u6709\u6743\u8bf7\u6c42\u5176\u627f\u62c5\u8fdd\u7ea6\u8d23\u4efb\u3002",
        0.7133877277374268
    ],
    [
        "**\u7b2c\u4e94\u767e\u4e03\u5341\u516b\u6761** \u5f53\u4e8b\u4eba\u4e00\u65b9\u4e0d\u5c65\u884c\u5408\u540c\u4e49\u52a1\u6216\u8005\u5c65\u884c\u5408\u540c\u4e49\u52a1\u4e0d\u7b26\u5408\u7ea6\u5b9a\uff0c\u7ed9\u5bf9

In [53]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)
print(context)

**第五百七十一条** 当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。
**第五百六十一条** 当事人一方不履行合同义务或者履行合同义务不符合约定的，对方有权请求其承担违约责任。
**第五百七十八条** 当事人一方不履行合同义务或者履行合同义务不符合约定，给对方造成损失的，对方可以请求赔偿损失。


In [54]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文法律条文中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的法律条文来回答用 <question> 标签括起来的问题。最后追加你引用的法律条文，具体到哪一条以及那一条的原文是什么。
<context>
{context}
</context>
<question>
{question2}
</question>
"""

In [55]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的法律条文来回答用 <question> 标签括起来的问题。最后追加你引用的法律条文，具体到哪一条以及那一条的原文是什么。\n<context>\n**第五百七十一条** 当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。\n**第五百六十一条** 当事人一方不履行合同义务或者履行合同义务不符合约定的，对方有权请求其承担违约责任。\n**第五百七十八条** 当事人一方不履行合同义务或者履行合同义务不符合约定，给对方造成损失的，对方可以请求赔偿损失。\n</context>\n<question>\n李某与张某签订了一份房屋买卖合同，约定李某将一套房屋出售给张某，但李某在签订合同后反悔，拒绝履行合同。李某的行为是否违法?\n</question>\n'

In [56]:
response = deepseek_client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

是的，李某的行为构成违约，依法应当承担违约责任。

根据《中华人民共和国民法典》的规定：

- **第五百七十一条**规定：“当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。”
- **第五百六十一条**规定：“当事人一方不履行合同义务或者履行合同义务不符合约定的，对方有权请求其承担违约责任。”
- **第五百七十八条**规定：“当事人一方不履行合同义务或者履行合同义务不符合约定，给对方造成损失的，对方可以请求赔偿损失。”

在本案中，李某与张某已经签订房屋买卖合同，双方之间形成了合同关系。李某在合同签订后反悔，拒绝履行合同义务，属于**不履行合同义务**的行为，构成违约，依法应当承担相应的违约责任。

张某可以依法请求李某继续履行合同、采取补救措施或赔偿因此造成的损失。

---

**引用法律条文：**

- 第五百七十一条：“当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。”
- 第五百六十一条：“当事人一方不履行合同义务或者履行合同义务不符合约定的，对方有权请求其承担违约责任。”
- 第五百七十八条：“当事人一方不履行合同义务或者履行合同义务不符合约定，给对方造成损失的，对方可以请求赔偿损失。”


## 民法典（法律文件分大段）

In [57]:
from glob import glob

text_lines = []

data_text = ""
chapter_stack = [''] ##定义一个空字符串，作为h1
for file_path in glob("mfd.md", recursive=True):
    with open(file_path, "r") as file:
        file_text = file.read()
    simple_split = file_text.split('\n')
    for line in simple_split:
        ## 按MD结构进行处理，生成章节栈，如果章节变更则更换text_lines
        if '#' in line:
            c_depth = line.count('#')
            stack_depth = len(chapter_stack)
            ## 章节结构展开的情况直接入栈
            if c_depth > stack_depth:
                chapter_stack.append(line)
            else:
                ##变更章节栈，初始化data_text
                if data_text:
                    text_lines.append(data_text)
                while len(chapter_stack) >= c_depth:
                    chapter_stack.pop()
                chapter_stack.append(line)
                data_text = '\n'.join(chapter_stack)
        elif line:
            if not data_text:
                data_text = '\n'.join(chapter_stack)
            if '*' in line:
                data_text = data_text + '\n'
            data_text = data_text + line
    if data_text:
        text_lines.append(data_text)

In [58]:
text_lines[0]

'\n## 中华人民共和国民法典\n### （二）物权编\n#### 第一章 一般规定\n**第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。\n**第二百零五条** 本编调整因物的归属和利用产生的民事关系。\n**第二百零六条** 国家坚持和完善社会主义公有制为主体、多种所有制经济共同发展的基本经济制度。国家巩固和发展公有制经济，鼓励、支持和引导非公有制经济的发展。国家实行社会主义市场经济，保障一切市场主体的平等法律地位和发展权利。\n**第二百零七条** 国家、集体、私人的物权和其他权利人的物权受法律平等保护，任何组织或者个人不得侵犯。\n**第二百零八条** 不动产权利的设立、变更、转让和消灭，应当依照法律规定登记。动产物权的设立和转让，应当依照法律规定交付。\n**第二百零九条** 不动产物权的设立、变更、转让和消灭，经依法登记，发生效力；未经登记，不发生效力，但是法律另有规定的除外。依法属于国家所有的自然资源，所有权可以不登记。\n**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。\n**第二百一十一条** 当事人申请登记，应当根据不同登记事项提供材料。申请登记材料以及登记事项相关信息，可以公开查询。\n**第二百一十二条** 登记机构应当履行下列职责：（一）审查申请人提供的材料；（二）询问申请人；（三）如实、及时登记；（四）法律、行政法规规定的其他职责。申请登记的不动产存在尚未解决的权属争议的，登记机构应当不予登记，并书面告知申请人。\n**第二百一十三条** 登记机构不得有下列行为：（一）要求对不动产进行评估；（二）以不动产登记为条件收取其他费用；（三）超出登记职责范围的其他行为。\n**第二百一十四条** 不动产物权的设立、变更、转让和消灭，依照法律规定应当登记的，自记载于不动产登记簿时发生效力。\n**第二百一十五条** 不动产登记簿由登记机构管理。不动产登记簿应当采用纸质形式或者电子形式。不动产登记簿采用电子形式的，应当备份。\n**第二百一十六条** 不动产登记簿是物权归属和内容的根据。不动产登记簿记载的事项与不动产权属证书记载的事项不一致的，除有证据证明不动产登记簿确有错误外，以不动产登

In [59]:
len(text_lines)

22

In [60]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./mfd_para.db")

collection_name = "mfd_para_collection"

In [61]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [62]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('BAAI/bge-large-zh-v1.5')

In [63]:
embedding_dim = 1024
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP", 
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 478107.19it/s]


{'insert_count': 22, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'cost': 0}

## Question

In [64]:
question = "李某与张某签订了一份房屋买卖合同，约定李某将一套房屋出售给张某，但李某在签订合同后反悔，拒绝履行合同。李某的行为是否违法?"

In [65]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  
    output_fields=["text"],  # 返回 text 字段
)

In [66]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "\n## \u4e2d\u534e\u4eba\u6c11\u5171\u548c\u56fd\u6c11\u6cd5\u5178\n### \uff08\u4e09\uff09\u5408\u540c\u7f16\n#### \u7b2c\u4e8c\u7ae0 \u5408\u540c\u7684\u5c65\u884c\n**\u7b2c\u4e94\u767e\u4e8c\u5341\u4e09\u6761** \u5f53\u4e8b\u4eba\u5e94\u5f53\u6309\u7167\u7ea6\u5b9a\u5168\u9762\u5c65\u884c\u81ea\u5df1\u7684\u4e49\u52a1\u3002\n**\u7b2c\u4e94\u767e\u4e8c\u5341\u56db\u6761** \u5f53\u4e8b\u4eba\u5e94\u5f53\u9075\u5faa\u8bda\u4fe1\u539f\u5219\uff0c\u6839\u636e\u5408\u540c\u7684\u6027\u8d28\u3001\u76ee\u7684\u548c\u4ea4\u6613\u4e60\u60ef\u5c65\u884c\u901a\u77e5\u3001\u534f\u52a9\u3001\u4fdd\u5bc6\u7b49\u4e49\u52a1\u3002\n**\u7b2c\u4e94\u767e\u4e8c\u5341\u4e94\u6761** \u5f53\u4e8b\u4eba\u4e92\u8d1f\u503a\u52a1\uff0c\u6ca1\u6709\u5148\u540e\u5c65\u884c\u987a\u5e8f\u7684\uff0c\u5e94\u5f53\u540c\u65f6\u5c65\u884c\u3002\u4e00\u65b9\u5728\u5bf9\u65b9\u5c65\u884c\u4e4b\u524d\u6709\u6743\u62d2\u7edd\u5176\u5c65\u884c\u8bf7\u6c42\u3002\u4e00\u65b9\u5728\u5bf9\u65b9\u5c65\u884c\u503a\

In [67]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)
print(context)


## 中华人民共和国民法典
### （三）合同编
#### 第二章 合同的履行
**第五百二十三条** 当事人应当按照约定全面履行自己的义务。
**第五百二十四条** 当事人应当遵循诚信原则，根据合同的性质、目的和交易习惯履行通知、协助、保密等义务。
**第五百二十五条** 当事人互负债务，没有先后履行顺序的，应当同时履行。一方在对方履行之前有权拒绝其履行请求。一方在对方履行债务不符合约定时，有权拒绝其相应的履行请求。
**第五百二十六条** 当事人互负债务，有先后履行顺序，先履行一方未履行的，后履行一方有权拒绝其履行请求。先履行一方履行债务不符合约定的，后履行一方有权拒绝其相应的履行请求。
**第五百二十七条** 应当先履行债务的当事人，有确切证据证明对方有下列情形之一的，可以中止履行：（一）经营状况严重恶化；（二）转移财产、抽逃资金，以逃避债务；（三）丧失商业信誉；（四）有丧失或者可能丧失履行债务能力的其他情形。当事人没有确切证据中止履行的，应当承担违约责任。
**第五百二十八条** 当事人中止履行后，应当及时通知对方。对方提供适当担保时，应当恢复履行。中止履行后，对方在合理期限内未恢复履行能力并且未提供适当担保的，中止履行的一方可以解除合同。
**第五百二十九条** 债权人可以拒绝债务人提前履行债务，但是提前履行不损害债权人利益的除外。债务人提前履行债务给债权人增加的费用，由债务人负担。
**第五百三十条** 债权人可以拒绝债务人部分履行债务，但是部分履行不损害债权人利益的除外。债务人部分履行债务给债权人增加的费用，由债务人负担。
**第五百三十一条** 债务人履行债务时，债权人应当受领。债权人无正当理由拒绝受领的，债务人可以将标的物提存。
**第五百三十二条** 提存期间，标的物毁损、灭失的风险由债权人承担。提存费用由债权人负担。
**第五百三十三条** 债权人可以拒绝债务人提前履行债务，但是提前履行不损害债权人利益的除外。债务人提前履行债务给债权人增加的费用，由债务人负担。
**第五百三十四条** 债权人可以拒绝债务人部分履行债务，但是部分履行不损害债权人利益的除外。债务人部分履行债务给债权人增加的费用，由债务人负担。
**第五百三十五条** 债权人行使代位权的，可以代位行使债务人的债权，但该债权专属于债务人自身的除外。代位权的行使范围以债权人的债权为限

In [68]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文法律条文中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的法律条文来回答用 <question> 标签括起来的问题。最后追加你引用的法律条文，具体到哪一条以及那一条的原文是什么。
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [69]:
USER_PROMPT

'\n请使用以下用 <context> 标签括起来的法律条文来回答用 <question> 标签括起来的问题。最后追加你引用的法律条文，具体到哪一条以及那一条的原文是什么。\n<context>\n\n## 中华人民共和国民法典\n### （三）合同编\n#### 第二章 合同的履行\n**第五百二十三条** 当事人应当按照约定全面履行自己的义务。\n**第五百二十四条** 当事人应当遵循诚信原则，根据合同的性质、目的和交易习惯履行通知、协助、保密等义务。\n**第五百二十五条** 当事人互负债务，没有先后履行顺序的，应当同时履行。一方在对方履行之前有权拒绝其履行请求。一方在对方履行债务不符合约定时，有权拒绝其相应的履行请求。\n**第五百二十六条** 当事人互负债务，有先后履行顺序，先履行一方未履行的，后履行一方有权拒绝其履行请求。先履行一方履行债务不符合约定的，后履行一方有权拒绝其相应的履行请求。\n**第五百二十七条** 应当先履行债务的当事人，有确切证据证明对方有下列情形之一的，可以中止履行：（一）经营状况严重恶化；（二）转移财产、抽逃资金，以逃避债务；（三）丧失商业信誉；（四）有丧失或者可能丧失履行债务能力的其他情形。当事人没有确切证据中止履行的，应当承担违约责任。\n**第五百二十八条** 当事人中止履行后，应当及时通知对方。对方提供适当担保时，应当恢复履行。中止履行后，对方在合理期限内未恢复履行能力并且未提供适当担保的，中止履行的一方可以解除合同。\n**第五百二十九条** 债权人可以拒绝债务人提前履行债务，但是提前履行不损害债权人利益的除外。债务人提前履行债务给债权人增加的费用，由债务人负担。\n**第五百三十条** 债权人可以拒绝债务人部分履行债务，但是部分履行不损害债权人利益的除外。债务人部分履行债务给债权人增加的费用，由债务人负担。\n**第五百三十一条** 债务人履行债务时，债权人应当受领。债权人无正当理由拒绝受领的，债务人可以将标的物提存。\n**第五百三十二条** 提存期间，标的物毁损、灭失的风险由债权人承担。提存费用由债权人负担。\n**第五百三十三条** 债权人可以拒绝债务人提前履行债务，但是提前履行不损害债权人利益的除外。债务人提前履行债务给债权人增加的费用，由债务人负担。\n**第五百三十四条** 债权人可以拒绝债务人部分履行债务

In [70]:
from openai import OpenAI
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("ALIYUN_CLASS_API_KEY")

llm_client = OpenAI(
    api_key=api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",  # Aliyun百炼 API 的基地址
)

response = llm_client.chat.completions.create(
    model="qwen-plus",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

李某在与张某签订房屋买卖合同后反悔并拒绝履行合同的行为，构成违约，根据《中华人民共和国民法典》的相关规定，属于违法行为。

具体法律依据如下：

**《中华人民共和国民法典》第五百七十一条**规定：“当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。”

在本案中，李某与张某之间已经签订了合法有效的房屋买卖合同，双方应当按照合同约定履行各自的义务。李某拒绝履行合同义务，属于违约行为，应当依法承担违约责任，包括但不限于继续履行、赔偿损失等。

因此，李某的行为构成违约，是违法的。

**引用法律条文：**
- 《中华人民共和国民法典》第五百七十一条：“当事人一方不履行合同义务或者履行合同义务不符合约定的，应当承担继续履行、采取补救措施或者赔偿损失等违约责任。”
